# RiemannianStaircaseResult

`RiemannianStaircaseResult` records the certified Burer–Monteiro solution, rounded matrix values, final rank, certificate information, and per-level objective and timing diagnostics.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415  
All Rights Reserved  
Authors: Frank Dellaert, et al. (see THANKS for the full author list)  
See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/certifiable/doc/RiemannianStaircaseResult.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam
from gtsam.symbol_shorthand import X

## Producing a result

Results are returned by `RiemannianStaircaseOptimizer.optimize()`. `hasRoundedSolution()` is true only after certification; calling `roundedValues()` otherwise raises. The per-level getters return NumPy-compatible vectors and keep wrapper code independent of C++ `std::vector<size_t>` details.

In [3]:
truth = [gtsam.Rot2.fromAngle(i * np.pi / 2.0) for i in range(4)]
graph = gtsam.NonlinearFactorGraph()
initial = gtsam.Values()
for i in range(4):
    j = (i + 1) % 4
    graph.add(gtsam.FrobeniusBetweenFactorRot2(
        X(i), X(j), truth[i].between(truth[j])
    ))
    initial.insert(X(i), gtsam.Rot2.fromAngle(i * np.pi / 2.0 + 0.01 * i).matrix().T)

alm = gtsam.AugmentedLagrangianParams()
alm.maxIterations = 80
alm.initialMuEq = 10.0
alm.muEqIncreaseRate = 2.0
params = gtsam.RiemannianStaircaseParams()
params.pMin = 2
params.pMax = 4
params.setAlmParams(alm)
result = gtsam.RiemannianStaircaseOptimizer(graph, initial, params).optimize()

In [4]:
print("certified:", result.certified)
print("final rank:", result.finalRank)
print("minimum eigenvalue / bound:", result.minEigenvalue)
print("total time:", result.totalTime)
print("ranks:", result.getRanksVisited())
print("costs:", result.getCostPerLevel())
print("certificate values:", result.getMinEigenvaluePerLevel())
print("local-solver times:", result.getNlpTimePerLevel())
print("verification times:", result.getVerifyTimePerLevel())

if result.hasRoundedSolution():
    rounded = result.roundedValues()
    print("rounded block shape:", rounded.atMatrix(X(0)).shape)

certified: True
final rank: 2
minimum eigenvalue / bound: -0.001
total time: 0.000364041
ranks: [2.]
costs: [6.43058725e-17]
certificate values: [-0.001]
local-solver times: [0.00032871]
verification times: [2.3875e-05]
rounded block shape: (2, 2)


Rounded Rot2 and Rot3 blocks still carry the component's common right-$O(D)$ gauge. Choose a common gauge, transpose each canonical block, and project to the nearest proper rotation before interpreting absolute orientations.